In [ ]:
# ==================================================
# 01. CONFIGURACIÓN DEL ENTORNO DE ENTRENAMIENTO
# ==================================================

from google.colab import drive
drive.mount("/content/drive")

import os
import sys
import json
import random
import numpy as np
import torch
import sentencepiece as spm

from pathlib import Path

# Directorio principal
BASE_DIR = Path(
    "/content/drive/MyDrive/Proyecto_traductor/traductor_optimizado"
)

DATA_DIR = BASE_DIR / "datasets" / "raw"
MODEL_DIR = BASE_DIR / "modelo"
CONFIG_DIR = BASE_DIR / "configuracion"
TOKENIZER_DIR = BASE_DIR / "tokenizador"

CHECKPOINT_DIR = BASE_DIR / "checkpoints"
RESULTS_DIR = BASE_DIR / "resultados"
GRAFICOS_DIR = RESULTS_DIR / "graficos" / "entrenamiento"

for directorio in [CHECKPOINT_DIR, RESULTS_DIR, GRAFICOS_DIR]:
    directorio.mkdir(parents=True, exist_ok=True)

# Reproducibilidad
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Dispositivo
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# Cargar configuración
with open(CONFIG_DIR / "modelo_config.json", "r") as f:
    MODEL_CONFIG = json.load(f)

# Cargar tokenizador
sp = spm.SentencePieceProcessor(
    model_file=str(TOKENIZER_DIR / "tokenizer.model")
)

# Importar arquitectura guardada
sys.path.insert(0, str(MODEL_DIR))

from transformer_model import TranslationTransformer

# Verificar archivos
archivos = [
    DATA_DIR / "train.tsv",
    DATA_DIR / "val.tsv",
    DATA_DIR / "test.tsv",
    MODEL_DIR / "transformer_model.py",
    CONFIG_DIR / "modelo_config.json",
    TOKENIZER_DIR / "tokenizer.model"
]

for archivo in archivos:
    assert archivo.exists(), f"No encontrado: {archivo}"

print("=" * 50)
print("ENTORNO DE ENTRENAMIENTO")
print("=" * 50)

print("PyTorch:", torch.__version__)
print("Dispositivo:", DEVICE)

if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "VRAM:",
        round(
            torch.cuda.get_device_properties(0).total_memory / 1024**3,
            2
        ),
        "GB"
    )

print("Vocabulario:", sp.vocab_size())
print("Longitud máxima:", MODEL_CONFIG["max_length"])
print("Archivos necesarios: OK")
print("Directorios de resultados: OK")
print("\nEntorno preparado correctamente.")

Mounted at /content/drive
ENTORNO DE ENTRENAMIENTO
PyTorch: 2.11.0+cpu
Dispositivo: cpu
Vocabulario: 16000
Longitud máxima: 80
Archivos necesarios: OK
Directorios de resultados: OK

Entorno preparado correctamente.


In [ ]:
# ==================================================
# 02. DATASET Y DATALOADER - VERSIÓN CORREGIDA
# ==================================================

import csv
import pandas as pd
import torch

from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

# 1. Cargar datasets
def cargar_tsv(ruta):
    return pd.read_csv(
        ruta,
        sep="\t",
        header=None,
        names=["en", "es", "id_original"],
        dtype=str,
        keep_default_na=False,
        quoting=csv.QUOTE_NONE
    )

train_df = cargar_tsv(DATA_DIR / "train.tsv")
val_df = cargar_tsv(DATA_DIR / "val.tsv")

print(f"Train: {len(train_df):,}")
print(f"Val: {len(val_df):,}")


# 2. Dataset
class TranslationDataset(Dataset):

    def __init__(self, df, tokenizer, max_length=80):
        self.en = df["en"].tolist()
        self.es = df["es"].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.en)

    def encode(self, text):
        tokens = self.tokenizer.encode(text, out_type=int)
        tokens = tokens[:self.max_length - 2]

        return (
            [self.tokenizer.bos_id()]
            + tokens
            + [self.tokenizer.eos_id()]
        )

    def __getitem__(self, idx):
        src = self.encode(self.en[idx])
        tgt = self.encode(self.es[idx])

        return (
            torch.tensor(src, dtype=torch.long),
            torch.tensor(tgt, dtype=torch.long)
        )


# 3. Padding dinámico
def collate_fn(batch):

    src_list, tgt_list = zip(*batch)

    src = pad_sequence(
        src_list,
        batch_first=True,
        padding_value=sp.pad_id()
    )

    tgt = pad_sequence(
        tgt_list,
        batch_first=True,
        padding_value=sp.pad_id()
    )

    return (
        src,
        tgt[:, :-1],
        tgt[:, 1:]
    )


# 4. Crear datasets
train_dataset = TranslationDataset(
    train_df, sp, MODEL_CONFIG["max_length"]
)

val_dataset = TranslationDataset(
    val_df, sp, MODEL_CONFIG["max_length"]
)


# 5. Crear DataLoaders
BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=0,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=0,
    pin_memory=True
)


# 6. Comprobación
src, tgt_input, tgt_expected = next(iter(train_loader))

assert tgt_input.shape == tgt_expected.shape
assert src.shape[0] == BATCH_SIZE

print("\n========== VALIDACIÓN ==========")
print("Lotes entrenamiento:", len(train_loader))
print("Lotes validación:", len(val_loader))
print("Encoder:", src.shape)
print("Decoder input:", tgt_input.shape)
print("Decoder target:", tgt_expected.shape)
print("\nDataLoaders creados correctamente.")

Train: 792,070
Val: 44,004

========== VALIDACIÓN ==========
Lotes entrenamiento: 24753
Lotes validación: 1376
Encoder: torch.Size([32, 36])
Decoder input: torch.Size([32, 35])
Decoder target: torch.Size([32, 35])

DataLoaders creados correctamente.


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


In [ ]:
print("Train loader:", len(train_loader))
print("Val loader:", len(val_loader))

src, tgt_input, tgt_expected = next(iter(train_loader))

print("Encoder:", src.shape)
print("Decoder input:", tgt_input.shape)
print("Decoder target:", tgt_expected.shape)

print("DataLoader funcionando correctamente.")

Train loader: 24753
Val loader: 1376
Encoder: torch.Size([32, 38])
Decoder input: torch.Size([32, 27])
Decoder target: torch.Size([32, 27])
DataLoader funcionando correctamente.


In [ ]:
# ==================================================
# 03. BENCHMARK DE ENTRENAMIENTO EN GPU
# ==================================================

import time
import torch
import torch.nn.functional as F

# Crear modelo temporal desde cero
benchmark_model = TranslationTransformer(
    MODEL_CONFIG
).to(DEVICE)

benchmark_model.train()

# Optimizador temporal
benchmark_optimizer = torch.optim.AdamW(
    benchmark_model.parameters(),
    lr=1e-4
)

# Precisión mixta para GPU T4
USE_AMP = DEVICE.type == "cuda"

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=USE_AMP
)

# Reiniciar estadísticas de memoria
if USE_AMP:
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()

tiempos = []
perdidas = []

iterador = iter(train_loader)

print("Iniciando benchmark de 5 pasos...\n")

for paso in range(5):

    src, tgt_input, tgt_expected = next(iterador)

    src = src.to(DEVICE, non_blocking=True)
    tgt_input = tgt_input.to(DEVICE, non_blocking=True)
    tgt_expected = tgt_expected.to(DEVICE, non_blocking=True)

    benchmark_optimizer.zero_grad(set_to_none=True)

    if USE_AMP:
        torch.cuda.synchronize()

    inicio = time.perf_counter()

    # Forward con precisión mixta
    with torch.autocast(
        device_type=DEVICE.type,
        dtype=torch.float16 if USE_AMP else torch.bfloat16,
        enabled=USE_AMP
    ):
        logits = benchmark_model(src, tgt_input)

        loss = F.cross_entropy(
            logits.reshape(-1, MODEL_CONFIG["vocab_size"]),
            tgt_expected.reshape(-1),
            ignore_index=MODEL_CONFIG["pad_id"]
        )

    # Backward
    scaler.scale(loss).backward()

    # Actualizar parámetros
    scaler.step(benchmark_optimizer)
    scaler.update()

    if USE_AMP:
        torch.cuda.synchronize()

    duracion = time.perf_counter() - inicio

    tiempos.append(duracion)
    perdidas.append(loss.item())

    print(
        f"Paso {paso + 1}/5 | "
        f"Loss: {loss.item():.4f} | "
        f"Tiempo: {duracion:.2f} s"
    )

# Estadísticas
tiempo_promedio = sum(tiempos[1:]) / len(tiempos[1:])

print("\n" + "=" * 50)
print("RESULTADOS DEL BENCHMARK")
print("=" * 50)

print(f"Batch size: {BATCH_SIZE}")
print(f"Tiempo promedio por paso: {tiempo_promedio:.3f} s")

if USE_AMP:
    memoria_maxima = (
        torch.cuda.max_memory_allocated() / 1024**3
    )
    print(f"VRAM máxima asignada: {memoria_maxima:.2f} GB")

print(
    f"Tiempo estimado por época (solo pasos GPU): "
    f"{len(train_loader) * tiempo_promedio / 3600:.2f} horas"
)

# Liberar modelo temporal
del benchmark_model
del benchmark_optimizer
del scaler
del logits
del loss

import gc
gc.collect()

if USE_AMP:
    torch.cuda.empty_cache()

print("\nBenchmark finalizado.")

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Iniciando benchmark de 5 pasos...

Paso 1/5 | Loss: 9.7345 | Tiempo: 3.64 s
Paso 2/5 | Loss: 9.4720 | Tiempo: 2.56 s
Paso 3/5 | Loss: 9.3059 | Tiempo: 1.45 s
Paso 4/5 | Loss: 9.1675 | Tiempo: 1.68 s
Paso 5/5 | Loss: 9.0531 | Tiempo: 1.98 s

RESULTADOS DEL BENCHMARK
Batch size: 32
Tiempo promedio por paso: 1.915 s
Tiempo estimado por época (solo pasos GPU): 13.16 horas

Benchmark finalizado.


In [ ]:
# ==================================================
# 04. PRUEBA DE ESTABILIDAD Y MEMORIA GPU
# ==================================================

import gc
import time
import torch
import torch.nn.functional as F

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

modelo_test = TranslationTransformer(MODEL_CONFIG).to(DEVICE)
modelo_test.train()

opt_test = torch.optim.AdamW(
    modelo_test.parameters(),
    lr=1e-4
)

scaler_test = torch.amp.GradScaler("cuda")

# Lote sintético de longitud máxima
B = 32
L = MODEL_CONFIG["max_length"]
V = MODEL_CONFIG["vocab_size"]

src_test = torch.randint(
    4, V, (B, L),
    device=DEVICE
)

tgt_test = torch.randint(
    4, V, (B, L),
    device=DEVICE
)

# BOS al inicio y EOS al final
src_test[:, 0] = MODEL_CONFIG["bos_id"]
src_test[:, -1] = MODEL_CONFIG["eos_id"]

tgt_test[:, 0] = MODEL_CONFIG["bos_id"]
tgt_test[:, -1] = MODEL_CONFIG["eos_id"]

tgt_in = tgt_test[:, :-1]
tgt_out = tgt_test[:, 1:]

torch.cuda.synchronize()
inicio = time.perf_counter()

opt_test.zero_grad(set_to_none=True)

with torch.autocast("cuda", dtype=torch.float16):
    logits = modelo_test(src_test, tgt_in)

    loss = F.cross_entropy(
        logits.reshape(-1, V),
        tgt_out.reshape(-1),
        ignore_index=MODEL_CONFIG["pad_id"]
    )

scaler_test.scale(loss).backward()
scaler_test.unscale_(opt_test)

grad_norm = torch.nn.utils.clip_grad_norm_(
    modelo_test.parameters(),
    max_norm=1.0
)

scaler_test.step(opt_test)
scaler_test.update()

torch.cuda.synchronize()
duracion = time.perf_counter() - inicio

memoria = torch.cuda.max_memory_allocated() / 1024**3
reservada = torch.cuda.max_memory_reserved() / 1024**3

print("=" * 48)
print("PRUEBA DE ESTABILIDAD - LONGITUD MÁXIMA")
print("=" * 48)

print(f"Batch size: {B}")
print(f"Longitud encoder: {L}")
print(f"Longitud decoder: {L - 1}")
print(f"Loss: {loss.item():.4f}")
print(f"Norma de gradientes: {grad_norm.item():.4f}")
print(f"Tiempo: {duracion:.3f} s")
print(f"VRAM máxima asignada: {memoria:.2f} GB")
print(f"VRAM máxima reservada: {reservada:.2f} GB")
print(f"Escala AMP: {scaler_test.get_scale():.0f}")

assert torch.isfinite(loss).item()
assert torch.isfinite(grad_norm).item()

print("\nPrueba completada correctamente.")

# Liberar recursos
del modelo_test, opt_test, scaler_test
del logits, loss, grad_norm
del src_test, tgt_test, tgt_in, tgt_out

gc.collect()
torch.cuda.empty_cache()

AttributeError: module 'torch._C' has no attribute '_cuda_resetPeakMemoryStats'

In [ ]:
# ==================================================
# 05. CONFIGURACIÓN DEL ENTRENAMIENTO - 15 ÉPOCAS
# ==================================================

import math
import json
import torch

from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR

# 1. Hiperparámetros
TRAIN_CONFIG = {
    "epochs": 15,
    "batch_size": 32,
    "gradient_accumulation_steps": 4,
    "learning_rate": 3e-4,
    "min_lr_ratio": 0.1,
    "warmup_ratio": 0.05,
    "weight_decay": 0.01,
    "max_grad_norm": 1.0,
    "label_smoothing": 0.0,
    "use_amp": True,
    "seed": 42
}

# 2. Batch efectivo
TRAIN_CONFIG["effective_batch_size"] = (
    TRAIN_CONFIG["batch_size"]
    * TRAIN_CONFIG["gradient_accumulation_steps"]
)

# 3. Calcular actualizaciones
steps_per_epoch = math.ceil(
    len(train_loader)
    / TRAIN_CONFIG["gradient_accumulation_steps"]
)

total_steps = (
    steps_per_epoch * TRAIN_CONFIG["epochs"]
)

warmup_steps = max(
    1,
    int(total_steps * TRAIN_CONFIG["warmup_ratio"])
)

TRAIN_CONFIG["steps_per_epoch"] = steps_per_epoch
TRAIN_CONFIG["total_steps"] = total_steps
TRAIN_CONFIG["warmup_steps"] = warmup_steps

# 4. Inicializar Transformer desde cero
model = TranslationTransformer(
    MODEL_CONFIG
).to(DEVICE)

# 5. Optimizador AdamW
optimizer = AdamW(
    model.parameters(),
    lr=TRAIN_CONFIG["learning_rate"],
    weight_decay=TRAIN_CONFIG["weight_decay"]
)

# 6. Scheduler: warmup + decaimiento lineal
def lr_lambda(step):

    if step < warmup_steps:
        return (step + 1) / warmup_steps

    progress = (
        (step - warmup_steps)
        / max(1, total_steps - warmup_steps)
    )

    return max(
        TRAIN_CONFIG["min_lr_ratio"],
        1.0 - progress * (
            1.0 - TRAIN_CONFIG["min_lr_ratio"]
        )
    )

scheduler = LambdaLR(
    optimizer,
    lr_lambda=lr_lambda
)

# 7. Precisión mixta FP16
scaler = torch.amp.GradScaler(
    "cuda",
    enabled=(
        TRAIN_CONFIG["use_amp"]
        and DEVICE.type == "cuda"
    )
)

# 8. Guardar configuración en Google Drive
train_config_path = (
    CONFIG_DIR / "entrenamiento_config.json"
)

with open(train_config_path, "w", encoding="utf-8") as f:
    json.dump(TRAIN_CONFIG, f, indent=4)

# 9. Mostrar resumen
print("=" * 55)
print("CONFIGURACIÓN DEL ENTRENAMIENTO")
print("=" * 55)

print("Épocas:", TRAIN_CONFIG["epochs"])
print("Batch físico:", TRAIN_CONFIG["batch_size"])
print("Batch efectivo:", TRAIN_CONFIG["effective_batch_size"])
print("Acumulación:", TRAIN_CONFIG["gradient_accumulation_steps"])
print("Learning rate máximo:", TRAIN_CONFIG["learning_rate"])
print("Optimizador: AdamW")
print("Scheduler: Warmup + decaimiento lineal")
print("Warmup steps:", warmup_steps)
print("Pasos por época:", steps_per_epoch)
print("Total de actualizaciones:", total_steps)
print("AMP:", scaler.is_enabled())

print(
    "\nParámetros entrenables:",
    f"{sum(p.numel() for p in model.parameters() if p.requires_grad):,}"
)

print("\nConfiguración guardada en:")
print(train_config_path)

print("\nModelo preparado para entrenamiento.")

In [ ]:
# ==================================================
# 06. CHECKPOINTS Y RECUPERACIÓN DEL ENTRENAMIENTO
# ==================================================

import json
import random
import numpy as np
import torch

from pathlib import Path

CHECKPOINT_DIR = BASE_DIR / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

LAST_CHECKPOINT = CHECKPOINT_DIR / "ultimo_checkpoint.pth"
BEST_CHECKPOINT = CHECKPOINT_DIR / "mejor_modelo.pth"
HISTORY_PATH = RESULTS_DIR / "historial_entrenamiento.json"

# Estado inicial del entrenamiento
start_epoch = 0
best_val_loss = float("inf")

history = {
    "epoch": [],
    "train_loss": [],
    "val_loss": [],
    "val_perplexity": [],
    "learning_rate": [],
    "epoch_time_seconds": []
}


def guardar_checkpoint(
    ruta,
    epoch,
    model,
    optimizer,
    scheduler,
    scaler,
    best_val_loss,
    history
):
    """Guarda todos los estados necesarios para reanudar."""

    checkpoint = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "best_val_loss": best_val_loss,
        "history": history,
        "model_config": MODEL_CONFIG,
        "train_config": TRAIN_CONFIG,

        # Estados de aleatoriedad
        "torch_rng_state": torch.get_rng_state(),
        "numpy_rng_state": np.random.get_state(),
        "python_rng_state": random.getstate()
    }

    if torch.cuda.is_available():
        checkpoint["cuda_rng_state"] = torch.cuda.get_rng_state_all()

    # Escritura temporal para evitar sobrescribir un
    # checkpoint válido con un archivo incompleto
    ruta = Path(ruta)
    temporal = ruta.with_suffix(".tmp")

    torch.save(checkpoint, temporal)
    temporal.replace(ruta)


def cargar_checkpoint(
    ruta,
    model,
    optimizer,
    scheduler,
    scaler
):
    """Restaura el entrenamiento desde un checkpoint."""

    checkpoint = torch.load(
        ruta,
        map_location=DEVICE,
        weights_only=False
    )

    if checkpoint["model_config"] != MODEL_CONFIG:
        raise ValueError(
            "La configuración del modelo no coincide."
        )

    if checkpoint["train_config"] != TRAIN_CONFIG:
        raise ValueError(
            "La configuración de entrenamiento no coincide."
        )

    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
    scaler.load_state_dict(checkpoint["scaler_state_dict"])

    torch.set_rng_state(
        checkpoint["torch_rng_state"].cpu()
    )

    np.random.set_state(checkpoint["numpy_rng_state"])
    random.setstate(checkpoint["python_rng_state"])

    if (
        torch.cuda.is_available()
        and "cuda_rng_state" in checkpoint
    ):
        torch.cuda.set_rng_state_all(
            [s.cpu() for s in checkpoint["cuda_rng_state"]]
        )

    return (
        checkpoint["epoch"] + 1,
        checkpoint["best_val_loss"],
        checkpoint["history"]
    )


def guardar_historial(history):
    """Guarda métricas para las gráficas del informe."""

    temporal = HISTORY_PATH.with_suffix(".tmp")

    with open(temporal, "w", encoding="utf-8") as f:
        json.dump(
            history,
            f,
            indent=4,
            ensure_ascii=False
        )

    temporal.replace(HISTORY_PATH)


# Detectar checkpoint existente
if LAST_CHECKPOINT.exists():

    print("Se encontró un checkpoint anterior.")
    print("Ruta:", LAST_CHECKPOINT)
    print(
        "No se cargará automáticamente. "
        "Primero debemos verificar si corresponde "
        "a este entrenamiento."
    )

else:
    print("No existen checkpoints anteriores.")
    print("El entrenamiento comenzará desde la época 1.")


print("\n" + "=" * 50)
print("SISTEMA DE CHECKPOINTS")
print("=" * 50)

print("Último checkpoint:", LAST_CHECKPOINT)
print("Mejor modelo:", BEST_CHECKPOINT)
print("Historial:", HISTORY_PATH)

print("\nFunciones disponibles:")
print("- guardar_checkpoint()")
print("- cargar_checkpoint()")
print("- guardar_historial()")

print("\nSistema preparado correctamente.")

No existen checkpoints anteriores.
El entrenamiento comenzará desde la época 1.

SISTEMA DE CHECKPOINTS
Último checkpoint: /content/drive/MyDrive/Proyecto_traductor/traductor_optimizado/checkpoints/ultimo_checkpoint.pth
Mejor modelo: /content/drive/MyDrive/Proyecto_traductor/traductor_optimizado/checkpoints/mejor_modelo.pth
Historial: /content/drive/MyDrive/Proyecto_traductor/traductor_optimizado/resultados/historial_entrenamiento.json

Funciones disponibles:
- guardar_checkpoint()
- cargar_checkpoint()
- guardar_historial()

Sistema preparado correctamente.


In [ ]:
# ==================================================
# 07. FUNCIONES DE ENTRENAMIENTO Y VALIDACIÓN
# ==================================================

import math
import time
import torch
import torch.nn.functional as F

from tqdm.auto import tqdm


def calcular_loss(logits, targets):
    """
    Cross-entropy ignorando tokens PAD.
    Devuelve la pérdida media y el número de tokens válidos.
    """

    vocab_size = logits.size(-1)

    loss = F.cross_entropy(
        logits.reshape(-1, vocab_size).float(),
        targets.reshape(-1),
        ignore_index=MODEL_CONFIG["pad_id"],
        reduction="sum",
        label_smoothing=TRAIN_CONFIG["label_smoothing"]
    )

    num_tokens = targets.ne(
        MODEL_CONFIG["pad_id"]
    ).sum().item()

    return loss, num_tokens


def entrenar_epoca(epoch):
    model.train()

    total_loss = 0.0
    total_tokens = 0
    accumulation_steps = TRAIN_CONFIG["gradient_accumulation_steps"]

    optimizer.zero_grad(set_to_none=True)

    progreso = tqdm(
        train_loader,
        desc=f"Entrenamiento | Época {epoch + 1}",
        leave=True
    )

    # Número de lotes del último grupo de acumulación
    total_batches = len(train_loader)

    for batch_idx, (src, tgt_input, tgt_expected) in enumerate(progreso):

        src = src.to(DEVICE, non_blocking=True)
        tgt_input = tgt_input.to(DEVICE, non_blocking=True)
        tgt_expected = tgt_expected.to(DEVICE, non_blocking=True)

        # Tamaño real del grupo de acumulación
        group_start = (batch_idx // accumulation_steps) * accumulation_steps
        group_size = min(
            accumulation_steps,
            total_batches - group_start
        )

        with torch.autocast(
            device_type=DEVICE.type,
            dtype=torch.float16 if DEVICE.type == "cuda" else torch.bfloat16,
            enabled=scaler.is_enabled()
        ):
            logits = model(src, tgt_input)

            loss_sum, num_tokens = calcular_loss(
                logits, tgt_expected
            )

            # Normalización por tokens del lote.
            # Se divide por el número de microbatches del grupo.
            loss = (
                loss_sum / max(1, num_tokens)
            ) / group_size

        scaler.scale(loss).backward()

        total_loss += loss_sum.detach().item()
        total_tokens += num_tokens

        # Actualizar al completar el grupo de acumulación
        is_update_step = (
            (batch_idx + 1) % accumulation_steps == 0
            or (batch_idx + 1) == total_batches
        )

        if is_update_step:

            scaler.unscale_(optimizer)

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=TRAIN_CONFIG["max_grad_norm"]
            )

            previous_scale = scaler.get_scale()

            scaler.step(optimizer)
            scaler.update()

            # Avanzar scheduler solo si AdamW actualizó los pesos
            if scaler.get_scale() >= previous_scale:
                scheduler.step()

            optimizer.zero_grad(set_to_none=True)

        if (batch_idx + 1) % 100 == 0:
            progreso.set_postfix({
                "loss": f"{total_loss / max(1, total_tokens):.4f}",
                "lr": f"{optimizer.param_groups[0]['lr']:.2e}"
            })

    return total_loss / max(1, total_tokens)


@torch.no_grad()
def validar_epoca(epoch):
    model.eval()

    total_loss = 0.0
    total_tokens = 0

    progreso = tqdm(
        val_loader,
        desc=f"Validación | Época {epoch + 1}",
        leave=False
    )

    for src, tgt_input, tgt_expected in progreso:

        src = src.to(DEVICE, non_blocking=True)
        tgt_input = tgt_input.to(DEVICE, non_blocking=True)
        tgt_expected = tgt_expected.to(DEVICE, non_blocking=True)

        with torch.autocast(
            device_type=DEVICE.type,
            dtype=torch.float16 if DEVICE.type == "cuda" else torch.bfloat16,
            enabled=scaler.is_enabled()
        ):
            logits = model(src, tgt_input)

            loss_sum, num_tokens = calcular_loss(
                logits, tgt_expected
            )

        total_loss += loss_sum.item()
        total_tokens += num_tokens

    val_loss = total_loss / max(1, total_tokens)

    # Evitar overflow en exp
    val_perplexity = math.exp(
        min(val_loss, 100)
    )

    return val_loss, val_perplexity


print("=" * 50)
print("FUNCIONES DE ENTRENAMIENTO")
print("=" * 50)
print("entrenar_epoca(): DEFINIDA")
print("validar_epoca(): DEFINIDA")
print("Acumulación de gradientes:", TRAIN_CONFIG["gradient_accumulation_steps"])
print("Gradient clipping:", TRAIN_CONFIG["max_grad_norm"])
print("Precisión mixta:", scaler.is_enabled())
print("Loss: Cross-entropy por token, ignorando PAD")
print("\nFunciones listas correctamente.")

FUNCIONES DE ENTRENAMIENTO
entrenar_epoca(): DEFINIDA
validar_epoca(): DEFINIDA
Acumulación de gradientes: 4
Gradient clipping: 1.0
Precisión mixta: True
Loss: Cross-entropy por token, ignorando PAD

Funciones listas correctamente.


In [ ]:
# ==================================================
# 08. ENTRENAMIENTO COMPLETO DEL TRANSFORMER
# ==================================================

import time
import math
import json
import torch

# 1. Recuperar entrenamiento anterior si existe
if LAST_CHECKPOINT.exists():

    print("Checkpoint encontrado. Recuperando entrenamiento...")

    start_epoch, best_val_loss, history = cargar_checkpoint(
        LAST_CHECKPOINT,
        model,
        optimizer,
        scheduler,
        scaler
    )

    print(f"Continuando desde la época {start_epoch + 1}")

else:
    print("Iniciando entrenamiento desde cero.")
    start_epoch = 0
    best_val_loss = float("inf")


# 2. Bucle principal
for epoch in range(start_epoch, TRAIN_CONFIG["epochs"]):

    print("\n" + "=" * 60)
    print(f"ÉPOCA {epoch + 1}/{TRAIN_CONFIG['epochs']}")
    print("=" * 60)

    inicio = time.perf_counter()

    # Entrenamiento
    train_loss = entrenar_epoca(epoch)

    # Validación
    val_loss, val_perplexity = validar_epoca(epoch)

    duracion = time.perf_counter() - inicio

    current_lr = optimizer.param_groups[0]["lr"]

    # 3. Registrar métricas
    history["epoch"].append(epoch + 1)
    history["train_loss"].append(float(train_loss))
    history["val_loss"].append(float(val_loss))
    history["val_perplexity"].append(float(val_perplexity))
    history["learning_rate"].append(float(current_lr))
    history["epoch_time_seconds"].append(float(duracion))

    # 4. Comprobar si es el mejor modelo
    es_mejor = val_loss < best_val_loss

    if es_mejor:
        best_val_loss = val_loss

    # 5. Guardar checkpoint de recuperación
    guardar_checkpoint(
        LAST_CHECKPOINT,
        epoch,
        model,
        optimizer,
        scheduler,
        scaler,
        best_val_loss,
        history
    )

    # 6. Guardar mejor modelo
    if es_mejor:
        guardar_checkpoint(
            BEST_CHECKPOINT,
            epoch,
            model,
            optimizer,
            scheduler,
            scaler,
            best_val_loss,
            history
        )

        print("Nuevo mejor modelo guardado.")

    # 7. Guardar historial
    guardar_historial(history)

    # 8. Mostrar resumen
    print("\n" + "-" * 60)
    print(f"RESULTADOS ÉPOCA {epoch + 1}")
    print("-" * 60)

    print(f"Train Loss:       {train_loss:.4f}")
    print(f"Validation Loss:  {val_loss:.4f}")
    print(f"Validation PPL:   {val_perplexity:.4f}")
    print(f"Learning Rate:    {current_lr:.8f}")
    print(f"Duración:         {duracion / 60:.2f} minutos")
    print(f"Mejor Val Loss:   {best_val_loss:.4f}")

    print("\nCheckpoint actualizado en Google Drive.")

print("\n" + "=" * 60)
print("ENTRENAMIENTO FINALIZADO")
print("=" * 60)

print("Épocas completadas:", len(history["epoch"]))
print("Mejor pérdida de validación:", best_val_loss)
print("Mejor modelo:", BEST_CHECKPOINT)

NameError: name 'LAST_CHECKPOINT' is not defined

In [ ]:

from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

BASE_DIR = Path(
    "/content/drive/MyDrive/Proyecto_traductor/traductor_optimizado"
)

ARCHIVOS = {
    "Historial": BASE_DIR / "resultados/historial_entrenamiento.json",
    "Mejor modelo": BASE_DIR / "checkpoints/mejor_modelo.pth",
    "Último checkpoint": BASE_DIR / "checkpoints/ultimo_checkpoint.pth",
    "Tokenizador": BASE_DIR / "tokenizador/tokenizer.model",
    "Arquitectura": BASE_DIR / "modelo/transformer_model.py",
    "Configuración": BASE_DIR / "configuracion/modelo_config.json",
}

print("=" * 65)
print("VERIFICACIÓN DE ARCHIVOS DEL TRANSFORMER")
print("=" * 65)

for nombre, ruta in ARCHIVOS.items():
    if ruta.is_file():
        print(f"[OK] {nombre}: {ruta.stat().st_size / (1024**2):.2f} MB")
    else:
        print(f"[FALTA] {nombre}: {ruta}")

assert all(ruta.is_file() for ruta in ARCHIVOS.values()), (
    "Faltan archivos. Revisa los checkpoints antes de continuar."
)

print("\nTodos los archivos necesarios están disponibles.")


In [ ]:

import json
import pandas as pd
from IPython.display import display

HISTORIAL_PATH = BASE_DIR / "resultados/historial_entrenamiento.json"

with open(HISTORIAL_PATH, "r", encoding="utf-8") as f:
    historial = json.load(f)

# Admite historial como lista o como diccionario con una lista interna.
if isinstance(historial, list):
    registros = historial
elif isinstance(historial, dict):
    candidatos = [
        v for v in historial.values()
        if isinstance(v, list) and (not v or isinstance(v[0], dict))
    ]
    if not candidatos:
        raise ValueError("Formato de historial no reconocido.")
    registros = candidatos[0]
else:
    raise ValueError("Formato de historial no reconocido.")

df = pd.DataFrame(registros)

columnas = [
    "epoch", "train_loss", "val_loss",
    "val_perplexity", "learning_rate", "epoch_time_seconds"
]

print("Columnas disponibles:", df.columns.tolist())
print("Épocas registradas:", len(df))

faltantes = [c for c in columnas if c not in df.columns]
if faltantes:
    raise ValueError(f"Columnas no encontradas: {faltantes}")

df = df[columnas].copy()
df = df.sort_values("epoch").reset_index(drop=True)

display(df.round(5))

assert len(df) == 15, (
    f"Se esperaban 15 épocas, pero se encontraron {len(df)}."
)

assert df["epoch"].tolist() == list(range(1, 16)), (
    "Las épocas registradas no corresponden a 1–15."
)

print("\nHistorial completo: 15 épocas verificadas.")


In [ ]:

mejor_idx = df["val_loss"].idxmin()
mejor = df.loc[mejor_idx]
ultima = df.iloc[-1]

print("=" * 60)
print("RESUMEN FINAL DEL ENTRENAMIENTO")
print("=" * 60)

print(f"Épocas completadas:       {len(df)}")
print(f"Mejor época:              {int(mejor['epoch'])}")
print(f"Mejor Validation Loss:    {mejor['val_loss']:.4f}")
print(f"Perplexity del mejor:     {mejor['val_perplexity']:.4f}")
print(f"Train Loss del mejor:     {mejor['train_loss']:.4f}")
print(f"Val Loss última época:    {ultima['val_loss']:.4f}")

tiempo_total = df["epoch_time_seconds"].sum() / 3600
print(f"Tiempo total registrado:  {tiempo_total:.2f} horas")

reduccion = (
    (df.iloc[0]["val_loss"] - mejor["val_loss"])
    / df.iloc[0]["val_loss"] * 100
)

print(f"Reducción Val Loss:       {reduccion:.2f}%")

resumen = {
    "epocas_completadas": int(len(df)),
    "mejor_epoca": int(mejor["epoch"]),
    "mejor_train_loss": float(mejor["train_loss"]),
    "mejor_val_loss": float(mejor["val_loss"]),
    "mejor_val_perplexity": float(mejor["val_perplexity"]),
    "ultima_val_loss": float(ultima["val_loss"]),
    "tiempo_total_horas": float(tiempo_total),
    "reduccion_val_loss_porcentaje": float(reduccion),
}

RESUMEN_PATH = BASE_DIR / "resultados/resumen_entrenamiento.json"

with open(RESUMEN_PATH, "w", encoding="utf-8") as f:
    json.dump(resumen, f, indent=4, ensure_ascii=False)

print(f"\nResumen guardado en:\n{RESUMEN_PATH}")


In [ ]:

import matplotlib.pyplot as plt

GRAFICOS_DIR = BASE_DIR / "resultados/graficos/entrenamiento"
GRAFICOS_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.figsize": (9, 5),
    "font.size": 11,
    "axes.grid": True,
    "grid.alpha": 0.25,
})

# Gráfica 1: Train Loss vs Validation Loss
fig, ax = plt.subplots()
ax.plot(df["epoch"], df["train_loss"], marker="o", label="Train Loss")
ax.plot(df["epoch"], df["val_loss"], marker="s", label="Validation Loss")
ax.axvline(
    int(mejor["epoch"]),
    linestyle="--",
    alpha=0.6,
    label=f"Mejor época: {int(mejor['epoch'])}"
)
ax.set(
    title="Evolución de la pérdida durante el entrenamiento",
    xlabel="Época",
    ylabel="Cross-Entropy Loss"
)
ax.set_xticks(df["epoch"])
ax.legend()
fig.tight_layout()
fig.savefig(GRAFICOS_DIR / "01_train_vs_validation_loss.png", dpi=300)
plt.show()
plt.close(fig)

# Gráfica 2: Validation Perplexity
fig, ax = plt.subplots()
ax.plot(df["epoch"], df["val_perplexity"], marker="o")
ax.set(
    title="Evolución de la perplexity de validación",
    xlabel="Época",
    ylabel="Validation Perplexity"
)
ax.set_xticks(df["epoch"])
fig.tight_layout()
fig.savefig(GRAFICOS_DIR / "02_validation_perplexity.png", dpi=300)
plt.show()
plt.close(fig)

# Gráfica 3: Learning Rate
fig, ax = plt.subplots()
ax.plot(df["epoch"], df["learning_rate"], marker="o")
ax.set(
    title="Evolución del Learning Rate",
    xlabel="Época",
    ylabel="Learning Rate"
)
ax.set_xticks(df["epoch"])
fig.tight_layout()
fig.savefig(GRAFICOS_DIR / "03_learning_rate.png", dpi=300)
plt.show()
plt.close(fig)

print("Tres gráficas guardadas correctamente en:")
print(GRAFICOS_DIR)


In [ ]:

import torch

CHECKPOINT_PATH = BASE_DIR / "checkpoints/mejor_modelo.pth"

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False
)

print("=" * 60)
print("INSPECCIÓN DEL MEJOR CHECKPOINT")
print("=" * 60)

if not isinstance(checkpoint, dict):
    raise TypeError("El checkpoint no tiene formato de diccionario.")

print("Claves disponibles:")
for clave in checkpoint.keys():
    print(" -", clave)

print("\nCheckpoint leído correctamente.")

# No asumir nombres de claves sin inspeccionar el archivo.
